# Preprocessing pipelines — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D007_C6_Preprocessing_Pipelines.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# Chapter 6: Preprocessing Pipelines

> **Data** wine_types.csv, volunteers.csv  (exercises use clinic_referrals.csv) · **Time** 45 min · **Needs first** C1 to C5

## What you will be able to do

1. Build a Pipeline of transformers ending in an estimator
2. Route numeric and categorical columns separately with ColumnTransformer
3. Cross-validate and grid-search a pipeline without leaking
4. Reach a fitted step's parameters through named_steps
5. Write a custom transformer with fit and transform
6. Save and reload a fitted pipeline

## The one idea to hold on to

A pipeline is not tidiness. It is the only structure that makes leakage difficult, because every step refits inside every fold whether you remember to or not.

In real-world machine learning projects, you need to apply multiple preprocessing steps in sequence. **Pipelines** provide a clean, reproducible way to chain these steps together while avoiding common pitfalls like data leakage.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# sklearn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# ═══ CHECKPOINT — "Why Use Pipelines?" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("Checkpoint restored. Ready for: Why Use Pipelines?")

## 1. Why Use Pipelines?

### The Problem: Manual Preprocessing

Without pipelines, you might write code like this:

```python
# WRONG WAY - prone to errors and data leakage
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Fits on ALL data

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)

model.fit(X_train, y_train)
model.predict(X_test)
```

**Problems:**
1. **Data Leakage**: Scaler sees test data before training
2. **Hard to Reproduce**: Many separate steps to remember
3. **Error-Prone**: Easy to forget a step or apply wrong order
4. **Deployment Issues**: Must replicate exact steps in production

### The Solution: Pipelines

```python
# CORRECT WAY - clean and safe
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y)

pipeline.fit(X_train, y_train)  # Scaler fits only on training data
pipeline.predict(X_test)  # Automatically applies same transformation
```

---

## 2. Understanding Data Leakage

**Data Leakage** occurs when information from outside the training dataset is used to create the model. This leads to overly optimistic performance estimates.

### Common Sources of Leakage in Preprocessing:

| Step | Leakage Risk | Solution |
|------|--------------|----------|
| Scaling | Using test data to calculate mean/std | Fit on train only |
| Imputation | Using test data to calculate fill values | Fit on train only |
| Feature Selection | Selecting features based on all data | Select using train only |
| Encoding | Calculating frequencies from all data | Calculate from train only |

### Correct Workflow:

```
1. Split data into train/test FIRST
2. Fit preprocessors on TRAINING data only
3. Transform both train and test using fitted preprocessors
4. Train model on preprocessed training data
5. Evaluate on preprocessed test data
```

In [ ]:
# TODO: compute `df`, `X`, `y` from `Type`
# Name your result: df, X, y

In [ ]:
# TODO: compute `scaler_wrong`, `X_scaled_wrong`, `model_wrong`
# Name your result: scaler_wrong, X_scaled_wrong, model_wrong, score_wrong

In [ ]:
# TODO: compute `scaler_correct`, `X_train_scaled`, `X_test_scaled`
# Name your result: scaler_correct, X_train_scaled, X_test_scaled, model_correct

---

## 3. Building Your First Pipeline

A **Pipeline** chains multiple transformers and optionally an estimator (model) together.

```python
Pipeline([
    ('step_name_1', Transformer1()),
    ('step_name_2', Transformer2()),
    ('model', Estimator())  # Optional, must be last
])
```

In [ ]:
# TODO: compute `simple_pipeline`
# Name your result: simple_pipeline

In [ ]:
# TODO: compute `y_pred`, `score`
# Name your result: y_pred, score

In [ ]:
# ═══ CHECKPOINT — "Accessing Pipeline Components" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
df = pd.read_csv('../../00-data/wine_types.csv')
X = df.drop('Type', axis=1)
y = df['Type']
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)
X_train_wrong, X_test_wrong, y_train, y_test = train_test_split(X_scaled_wrong, y, test_size=0.2, random_state=42)
model_wrong = LogisticRegression(max_iter=1000)
model_wrong.fit(X_train_wrong, y_train)
score_wrong = model_wrong.score(X_test_wrong, y_test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_correct = StandardScaler()
X_train_scaled = scaler_correct.fit_transform(X_train)
X_test_scaled = scaler_correct.transform(X_test)
model_correct = LogisticRegression(max_iter=1000)
model_correct.fit(X_train_scaled, y_train)
score_correct = model_correct.score(X_test_scaled, y_test)
simple_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000))])
simple_pipeline.fit(X_train, y_train)
y_pred = simple_pipeline.predict(X_test)
score = simple_pipeline.score(X_test, y_test)

print("Checkpoint restored. Ready for: Accessing Pipeline Components")

### Accessing Pipeline Components

You can access individual steps and their fitted parameters.

In [ ]:
# TODO: write the code for this step
# Name your result: scaler, model

---

## 4. ColumnTransformer: Handling Mixed Data Types

Real datasets often have both numerical and categorical columns that need different preprocessing.

**ColumnTransformer** applies different transformers to different columns.

```python
ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_columns),
    ('categorical', categorical_pipeline, categorical_columns)
])
```

In [ ]:
# TODO: compute `mixed_df`
# Name your result: mixed_df

In [ ]:
# TODO: compute `numeric_features`, `categorical_features`, `numeric_transformer`
# Name your result: numeric_features, categorical_features, numeric_transformer, categorical_transformer

In [ ]:
# TODO: compute `X_mixed`, `y_mixed`, `X_transformed` from `hired`, `cat`, `onehot`
# Name your result: X_mixed, y_mixed, X_transformed, cat_encoder

---

## 5. Complete Pipeline with Model

Now let's combine the ColumnTransformer with a model in a full pipeline.

In [ ]:
# TODO: compute `full_pipeline`, `new_data`, `predictions`
# Name your result: full_pipeline, new_data, predictions

---

## 6. Pipelines with Cross-Validation

The real power of pipelines shines with cross-validation. Each fold properly fits the preprocessors on training data only.

In [ ]:
# TODO: compute `cv_pipeline`, `cv_scores`
# Name your result: cv_pipeline, cv_scores

---

## 7. Pipelines with Grid Search

You can tune hyperparameters of any step in the pipeline using GridSearchCV.

Use the naming convention: `step_name__parameter_name`

In [ ]:
# TODO: compute `grid_pipeline`, `param_grid`, `grid_search`
# Name your result: grid_pipeline, param_grid, grid_search

In [14]:
# You can even tune which scaler to use!
from sklearn.preprocessing import RobustScaler

# Create parameter grid that includes different preprocessors
param_grid_advanced = [
    {
        'scaler': [StandardScaler()],
        'classifier__C': [0.1, 1.0, 10.0]
    },
    {
        'scaler': [MinMaxScaler()],
        'classifier__C': [0.1, 1.0, 10.0]
    },
    {
        'scaler': [RobustScaler()],
        'classifier__C': [0.1, 1.0, 10.0]
    }
]

grid_search_advanced = GridSearchCV(grid_pipeline, param_grid_advanced, cv=5)
grid_search_advanced.fit(X, y)

print("Advanced Grid Search Results:")
print(f"  Best Score: {grid_search_advanced.best_score_:.4f}")
print(f"  Best Scaler: {grid_search_advanced.best_params_['scaler'].__class__.__name__}")
print(f"  Best C: {grid_search_advanced.best_params_['classifier__C']}")

Advanced Grid Search Results:
  Best Score: 0.9889
  Best Scaler: RobustScaler
  Best C: 1.0


---

## 8. Best Practices for Pipelines

### Do:

| Practice | Reason |
|----------|--------|
| Always split data FIRST | Prevents data leakage |
| Use meaningful step names | Makes debugging easier |
| Use pipelines with cross-validation | Ensures proper fitting per fold |
| Save the fitted pipeline | For deployment and reproducibility |

### Don't:

| Anti-Pattern | Problem |
|--------------|----------|
| Fit preprocessors on all data | Data leakage |
| Manual preprocessing before CV | Inconsistent results |
| Forget to handle new categories | Prediction failures |

### Saving and Loading Pipelines

In [15]:
import joblib

# Train a final pipeline
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, C=1.0))
])
final_pipeline.fit(X_train, y_train)

# Save the pipeline
# joblib.dump(final_pipeline, 'wine_classifier_pipeline.pkl')

# Later, load and use
# loaded_pipeline = joblib.load('wine_classifier_pipeline.pkl')
# predictions = loaded_pipeline.predict(new_data)

print("Pipeline can be saved with joblib.dump() and loaded with joblib.load()")
print("This saves the entire pipeline including fitted preprocessors!")

Pipeline can be saved with joblib.dump() and loaded with joblib.load()
This saves the entire pipeline including fitted preprocessors!


---

## 9. Real-World Example: Complete Preprocessing Pipeline

In [ ]:
# TODO: compute `volunteers`
# Name your result: volunteers

In [17]:
# Prepare features (simplified for demonstration)
# Create a binary target: high engagement (hits > median)
volunteers['high_engagement'] = (volunteers['hits'] > volunteers['hits'].median()).astype(int)

# Select features
feature_cols = ['vol_requests', 'category_desc', 'recurrence_type']
X_vol = volunteers[feature_cols].copy()
y_vol = volunteers['high_engagement']

print("Features:")
print(X_vol.dtypes)
print(f"\nTarget distribution: {y_vol.value_counts().to_dict()}")

Features:
vol_requests        int64
category_desc      object
recurrence_type    object
dtype: object

Target distribution: {0: 333, 1: 332}


In [18]:
# Define preprocessing for each column type
numeric_cols = ['vol_requests']
categorical_cols = ['category_desc', 'recurrence_type']

# Numeric: impute with median, then scale
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: impute with constant, then one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

# Full pipeline with model
complete_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

print("Complete Pipeline Structure:")
print(complete_pipeline)

Complete Pipeline Structure:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['vol_requests']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Unknown',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                

In [19]:
# Evaluate with cross-validation
cv_scores = cross_val_score(complete_pipeline, X_vol, y_vol, cv=5, scoring='accuracy')

print("Cross-Validation Results:")
print(f"  Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

Cross-Validation Results:
  Accuracy: 0.5744 (+/- 0.0982)


---

## Summary

In this chapter, you learned:

1. **Why Pipelines**: Prevent data leakage, improve reproducibility
2. **Basic Pipeline**: Chain transformers and models together
3. **ColumnTransformer**: Handle mixed data types differently
4. **Cross-Validation**: Pipelines ensure proper fitting per fold
5. **Grid Search**: Tune any parameter in the pipeline
6. **Best Practices**: Split first, use meaningful names, save pipelines

### Complete Workflow:

```python
# 1. Define preprocessing for each column type
numeric_transformer = Pipeline([...])
categorical_transformer = Pipeline([...])

# 2. Combine with ColumnTransformer
preprocessor = ColumnTransformer([...])

# 3. Create full pipeline with model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', Model())
])

# 4. Split data and evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y)
pipeline.fit(X_train, y_train)
score = pipeline.score(X_test, y_test)
```

### What's Next?

- **Chapter 7**: Case Study - Apply all preprocessing techniques to a real problem

---

In [ ]:
# ═══ CHECKPOINT — "Exercises" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
df = pd.read_csv('../../00-data/wine_types.csv')
X = df.drop('Type', axis=1)
y = df['Type']
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)
X_train_wrong, X_test_wrong, y_train, y_test = train_test_split(X_scaled_wrong, y, test_size=0.2, random_state=42)
model_wrong = LogisticRegression(max_iter=1000)
model_wrong.fit(X_train_wrong, y_train)
score_wrong = model_wrong.score(X_test_wrong, y_test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_correct = StandardScaler()
X_train_scaled = scaler_correct.fit_transform(X_train)
X_test_scaled = scaler_correct.transform(X_test)
model_correct = LogisticRegression(max_iter=1000)
model_correct.fit(X_train_scaled, y_train)
score_correct = model_correct.score(X_test_scaled, y_test)
simple_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000))])
simple_pipeline.fit(X_train, y_train)
y_pred = simple_pipeline.predict(X_test)
score = simple_pipeline.score(X_test, y_test)
scaler = simple_pipeline.named_steps['scaler']
model = simple_pipeline.named_steps['classifier']
mixed_df = pd.DataFrame({'age': [25, 30, np.nan, 45, 50], 'salary': [50000, 60000, 70000, np.nan, 90000], 'department': ['Sales', 'Engineering', 'Sales', 'HR', 'Engineering'], 'education': ['Bachelor', 'Master', 'PhD', 'Bachelor', 'Master'], 'hired': [1, 1, 0, 1, 0]})
numeric_features = ['age', 'salary']
categorical_features = ['department', 'education']
numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)])
X_mixed = mixed_df.drop('hired', axis=1)
y_mixed = mixed_df['hired']
X_transformed = preprocessor.fit_transform(X_mixed)
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_features = cat_encoder.get_feature_names_out(categorical_features)
all_features = list(numeric_features) + list(cat_features)
full_pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression())])
full_pipeline.fit(X_mixed, y_mixed)
new_data = pd.DataFrame({'age': [35, np.nan], 'salary': [75000, 80000], 'department': ['Marketing', 'Sales'], 'education': ['Bachelor', 'PhD']})
predictions = full_pipeline.predict(new_data)
cv_pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000))])
cv_scores = cross_val_score(cv_pipeline, X, y, cv=5, scoring='accuracy')
grid_pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000))])
param_grid = {'classifier__C': [0.1, 1.0, 10.0], 'classifier__solver': ['lbfgs', 'liblinear']}
grid_search = GridSearchCV(grid_pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X, y)
from sklearn.preprocessing import RobustScaler
param_grid_advanced = [{'scaler': [StandardScaler()], 'classifier__C': [0.1, 1.0, 10.0]}, {'scaler': [MinMaxScaler()], 'classifier__C': [0.1, 1.0, 10.0]}, {'scaler': [RobustScaler()], 'classifier__C': [0.1, 1.0, 10.0]}]
grid_search_advanced = GridSearchCV(grid_pipeline, param_grid_advanced, cv=5)
grid_search_advanced.fit(X, y)
import joblib
final_pipeline = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000, C=1.0))])
final_pipeline.fit(X_train, y_train)
volunteers = pd.read_csv('../../00-data/volunteers.csv')
volunteers['high_engagement'] = (volunteers['hits'] > volunteers['hits'].median()).astype(int)
feature_cols = ['vol_requests', 'category_desc', 'recurrence_type']
X_vol = volunteers[feature_cols].copy()
y_vol = volunteers['high_engagement']
numeric_cols = ['vol_requests']
categorical_cols = ['category_desc', 'recurrence_type']
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', numeric_pipeline, numeric_cols), ('cat', categorical_pipeline, categorical_cols)])
complete_pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])
cv_scores = cross_val_score(complete_pipeline, X_vol, y_vol, cv=5, scoring='accuracy')

print("Checkpoint restored. Ready for: Exercises")

## Exercises

### Exercise 1: Build a Complete Pipeline

Using the wine dataset:
1. Create a pipeline with StandardScaler and RandomForestClassifier
2. Use 5-fold cross-validation to evaluate
3. Compare with a pipeline using MinMaxScaler

In [20]:
# Your code here


### Exercise 2: Grid Search with Pipeline

1. Create a pipeline with StandardScaler and DecisionTreeClassifier
2. Use GridSearchCV to find the best `max_depth` (try 3, 5, 7, None)
3. Report the best parameters and score

In [21]:
# Your code here


### Exercise 3: Mixed Data Pipeline

Load the UFO sightings dataset and:
1. Select 2 numeric and 2 categorical features
2. Create appropriate preprocessing pipelines for each type
3. Combine with ColumnTransformer
4. Add a LogisticRegression model
5. Evaluate with cross-validation

In [22]:
# Your code here


---

## How this is used in real work

This is the deliverable. A trained model is a pipeline object; anything less is a notebook someone has to re-run by hand. `joblib.dump(pipeline)` is what gets deployed, and `GridSearchCV(pipeline, ...)` is how the hyperparameters of the preprocessing get tuned alongside the model's.

## Where this goes next

**C7** applies the whole sequence end to end. **C8** decides whether the result is any good.